In [3]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter



In [5]:
# Configurar estilo de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set(font_scale=1.1)
colors = sns.color_palette('viridis', 10)

# 1. Cargar los datos
df = pd.read_excel('C:\\Users\\sandr\\Documents\\scrp_tiktok_tfg\\analysis\\nlp\\products and brands detection\\resultados_finales.xlsx')


In [6]:
has_brands = df['mentioned_brands'].apply(lambda x: len(str(x)) > 2).sum()
has_products = df['mentioned_products'].apply(lambda x: len(str(x)) > 2).sum()
has_both = df[df['mentioned_brands'].apply(lambda x: len(str(x)) > 2) & 
              df['mentioned_products'].apply(lambda x: len(str(x)) > 2)].shape[0]

print(f"\nVídeos con marcas mencionadas: {has_brands} ({has_brands/df.shape[0]:.1%})")
print(f"Vídeos con productos mencionados: {has_products} ({has_products/df.shape[0]:.1%})")
print(f"Vídeos con ambos mencionados: {has_both} ({has_both/df.shape[0]:.1%})")



Vídeos con marcas mencionadas: 1146 (100.0%)
Vídeos con productos mencionados: 763 (66.6%)
Vídeos con ambos mencionados: 763 (66.6%)


In [7]:
# Función para convertir strings de listas a listas reales
def parse_list_string(x):
    if isinstance(x, str):
        try:
            # Eliminar espacios adicionales y comillas inconsistentes
            x = x.replace("'", '"').replace(' "', '"').replace('" ', '"')
            # Intentar parsear como JSON primero
            try:
                return ast.literal_eval(x)
            except:
                # Si falla, intentar limpiar más y volver a intentar
                x = x.strip('[]').split(',')
                return [item.strip().strip("'\"") for item in x if item.strip()]
        except (ValueError, SyntaxError):
            return []
    elif isinstance(x, list):
        return x
    else:
        return []

# Aplicar la conversión a las columnas relevantes
df['mentioned_brands'] = df['mentioned_brands'].apply(parse_list_string)
df['mentioned_products'] = df['mentioned_products'].apply(parse_list_string)
df['product_brands'] = df['product_brands'].apply(parse_list_string)

# 4. Eliminar filas donde tanto brands como products están vacíos
df_clean = df[(df['mentioned_brands'].apply(len) > 0) | (df['mentioned_products'].apply(len) > 0)].copy()
print(f"\nFilas eliminadas por no tener marcas ni productos: {df.shape[0] - df_clean.shape[0]}")

# 5. Eliminar duplicados de productos en un mismo video
def remove_product_duplicates(row):
    if len(row['mentioned_products']) > 0:
        # Convertir a conjunto para eliminar duplicados
        row['mentioned_products'] = list(set(row['mentioned_products']))
    return row

df_clean = df_clean.apply(remove_product_duplicates, axis=1)


Filas eliminadas por no tener marcas ni productos: 0


In [8]:
brand_mapping = {
    'ROSE INC': 'Rose Inc',
    'Charlotte Tilbury': 'Charlotte Tilbury',
    'NARS': 'NARS',
    'tarte': 'Tarte',
    'Tarte': 'Tarte',
    'Drunk Elephant': 'Drunk Elephant',
    'Benefit Cosmetics': 'Benefit Cosmetics',
    'MAC Cosmetics': 'MAC Cosmetics',
    'Fenty Beauty': 'Fenty Beauty',
    'LANEIGE': 'Laneige',
    'Isle of Paradise': 'Isle of Paradise',
    'goop': 'Goop',
    'Rare Beauty by Selena Gomez': 'Rare Beauty'
}

# Función para normalizar marcas
def normalize_brands(brands_list):
    normalized = []
    for brand in brands_list:
        # Buscar en el mapping o mantener el original
        normalized.append(brand_mapping.get(brand, brand))
    return normalized

# Aplicar normalización
df_clean['mentioned_brands_norm'] = df_clean['mentioned_brands'].apply(normalize_brands)


In [9]:
# 7. Crear transacciones para el análisis de cesta de compra
def create_transaction(row):
    transaction = []
    
    # Añadir productos con prefijo
    for product in row['mentioned_products']:
        if product:  # Solo añadir si no está vacío
            # Simplificar nombres muy largos para mejor visualización
            if len(product) > 40:
                product = product[:37] + '...'
            transaction.append(f"PROD_{product}")
    
    # Añadir marcas normalizadas con prefijo
    for brand in row['mentioned_brands_norm']:
        if brand:  # Solo añadir si no está vacío
            transaction.append(f"BRAND_{brand}")
    
    return transaction

# Crear columna de transacciones
df_clean['transaction'] = df_clean.apply(create_transaction, axis=1)

# Eliminar transacciones vacías
df_clean = df_clean[df_clean['transaction'].apply(len) > 0]

print("\nEjemplos de transacciones:")
for i, trans in enumerate(df_clean['transaction'].head(3)):
    print(f"Video {i+1}: {trans}")


Ejemplos de transacciones:
Video 1: ['PROD_Lip Cream Longwearing Matte Liquid Li...', 'PROD_Besties Iconic Makeup Sponge and Clea...', 'PROD_Soy Hydrating Gentle Face Cleanser', 'PROD_Total Cleans"r Remove-It-All Cleanser...', 'PROD_Slaai™  Makeup-Melting Butter Cleanser', 'PROD_Softlight Clean Dewy Hydrating Concealer', 'BRAND_Rose Inc']
Video 2: ['BRAND_Drunk Elephant']
Video 3: ['BRAND_Isle of Paradise']


In [ ]:
# Market Basket Analysis simplificado: 
# Solo analizando PARES de marcas que aparecen en el mismo video

import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
import ast
from collections import defaultdict
import itertools

# 1. Cargar datos
df = pd.read_excel('resultados_finales.xlsx')
print(f"Datos cargados: {df.shape[0]} videos")

# 2. Convertir strings a listas si es necesario
def parse_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    elif isinstance(x, list):
        return x
    else:
        return []

df['mentioned_brands'] = df['mentioned_brands'].apply(parse_list)

# 3. Crear transacciones basadas en URL de video
# Cada URL es una transacción, con las marcas mencionadas como items
transactions = {}
for idx, row in df.iterrows():
    url = row['video_url']
    brands = row['mentioned_brands']
    
    # Inicializar si es la primera vez que vemos este URL
    if url not in transactions:
        transactions[url] = set()
    
    # Añadir marcas a la transacción (set garantiza unicidad)
    for brand in brands:
        if brand and isinstance(brand, str):  # Verificar que sea string válido
            transactions[url].add(brand)

# 4. Filtrar para quedarnos solo con transacciones con al menos 2 marcas
valid_transactions = {url: brands for url, brands in transactions.items() if len(brands) >= 2}

# 5. Calcular frecuencia de pares de marcas directamente
pair_counts = defaultdict(int)
total_transactions = len(valid_transactions)

# Recorrer cada transacción y contar pares de marcas
for brands in valid_transactions.values():
    # Generar todos los pares posibles de marcas en este video
    for brand1, brand2 in itertools.combinations(sorted(brands), 2):
        # Crear un par ordenado para consistencia
        pair = (brand1, brand2)
        pair_counts[pair] += 1

# 6. Calcular soporte para cada par (porcentaje de videos donde aparecen juntas)
pair_support = {pair: count/total_transactions for pair, count in pair_counts.items()}

# 7. Calcular confianza y lift
# Primero contamos la frecuencia individual de cada marca
brand_counts = defaultdict(int)
for brands in valid_transactions.values():
    for brand in brands:
        brand_counts[brand] += 1

# Calculamos confianza y lift para cada par
pair_metrics = []
for pair, count in pair_counts.items():
    brand1, brand2 = pair
    support = pair_support[pair]
    
    # Confianza: P(B|A) = P(A y B) / P(A)
    confidence_1_2 = count / brand_counts[brand1]
    confidence_2_1 = count / brand_counts[brand2]
    
    # Lift: P(A y B) / (P(A) * P(B))
    lift = (count / total_transactions) / ((brand_counts[brand1] / total_transactions) * (brand_counts[brand2] / total_transactions))
    
    pair_metrics.append({
        'Marca1': brand1,
        'Marca2': brand2,
        'Soporte': support,
        'Confianza_1_2': confidence_1_2,  # P(Marca2|Marca1)
        'Confianza_2_1': confidence_2_1,  # P(Marca1|Marca2)
        'Lift': lift,
        'Frecuencia': count
    })

# 8. Convertir a DataFrame y ordenar por lift descendente
results_df = pd.DataFrame(pair_metrics)
results_df = results_df.sort_values('Lift', ascending=False)

# 9. Guardar resultados
results_df.to_excel('pares_marcas_mencionadas_juntas.xlsx', index=False)

# 10. Mostrar top 20 pares con mayor lift
print("\nTop 20 pares de marcas más frecuentemente mencionadas juntas (ordenados por lift):")
print("=" * 80)
print(f"Total de videos analizados: {total_transactions}")
print(f"Total de pares de marcas encontrados: {len(pair_metrics)}")
print("=" * 80)

# Formatear para mejor visualización
top_pairs = results_df.head(20)
for idx, row in top_pairs.iterrows():
    print(f"{row['Marca1']} y {row['Marca2']}:")
    print(f"  Frecuencia: {row['Frecuencia']} videos")
    print(f"  Soporte: {row['Soporte']:.3f} ({row['Soporte']*100:.1f}% de los videos)")
    print(f"  Confianza: {row['Confianza_1_2']:.2f} / {row['Confianza_2_1']:.2f}")
    print(f"  Lift: {row['Lift']:.2f}")
    print("-" * 40)

# 11. Mostrar estadísticas de las marcas individuales más mencionadas
print("\nMarcas más frecuentemente mencionadas:")
top_brands = sorted(brand_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for brand, count in top_brands:
    print(f"{brand}: {count} videos ({count/total_transactions*100:.1f}%)")

print("\n=== INTERPRETACIÓN ===")
print("- Soporte: % de videos donde ambas marcas aparecen juntas")
print("- Confianza_1_2: % de veces que la Marca2 aparece cuando se menciona la Marca1")
print("- Confianza_2_1: % de veces que la Marca1 aparece cuando se menciona la Marca2")
print("- Lift > 1: Las marcas tienden a aparecer juntas más de lo esperado por casualidad")
print("  (mayor lift = asociación más fuerte)")

Datos cargados: 1146 videos
Total de videos (URLs) analizados: 183
Videos con al menos una marca: 183
Videos con múltiples marcas: 173
Total de marcas únicas: 83

Aplicando Apriori con soporte mínimo: 0.01


C:\Users\sandr\AppData\Local\Temp\ipykernel_28408\1413138042.py:61: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  basket.iloc[i][brand] = 1
c:\Users\sandr\anaconda3\envs\MachineLearning\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py


Se encontraron 446762 reglas de asociación

Top 10 reglas por lift:
                                             Antecedentes  \
398978  Charlotte Tilbury, DUO, Dior Beauty, Huda Beau...   
446664  Rare Beauty by Selena Gomez, MAKEUP BY MARIO, ...   
328165      Drunk Elephant, DUO, Huda Beauty, Dior Beauty   
52012                           Yves Saint Laurent, tarte   
52013                           Yves Saint Laurent, Tarte   
52016                                   tarte, YSL Beauty   
52017                                   YSL Beauty, Tarte   
398957  Drunk Elephant, LANEIGE, Rare Beauty by Selena...   
398975  Charlotte Tilbury, DUO, Rare Beauty by Selena ...   
398976  Charlotte Tilbury, DUO, Rare Beauty by Selena ...   

                                             Consecuentes  Soporte  Confianza  \
398978  Drunk Elephant, Rare Beauty by Selena Gomez, L...    0.011        1.0   
446664  Drunk Elephant, tarte, Charlotte Tilbury, Rare...    0.011        1.0   
328165         R

In [ ]:
# si hay el mismo producto en un mismo video eliminar para q solo haya una aparición
# puede a ver la misma marca varias veces en un mismo video (puede que compren diferente productos de la misma marca)

In [ ]:
# DEFINICÓN DE TRANSACCIONES: 
# Las marcas son compras de productos de esa marca (aunque no se sepa el nombre del producto)
# Las marcas y productos que pertenecen a un mismo video significa que han sido comprados juntos


In [26]:
# Market Basket Analysis simplificado: 
# Transacciones basadas exclusivamente en URL de video

import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
import ast
from collections import defaultdict

# 1. Cargar datos
df = pd.read_excel('C:\\Users\\sandr\\Documents\\scrp_tiktok_tfg\\analysis\\nlp\\products and brands detection\\resultados_finales.xlsx')

print(f"Datos cargados: {df.shape[0]} videos")

# 2. Convertir strings a listas si es necesario
def parse_list(x):
    if isinstance(x, str):
        try:
            # Manejar diferentes formatos de listas en string
            return ast.literal_eval(x)
        except:
            return []
    elif isinstance(x, list):
        return x
    else:
        return []

df['mentioned_brands'] = df['mentioned_brands'].apply(parse_list)

# 3. Crear transacciones basadas en URL de video
# Cada URL es una transacción, con las marcas mencionadas como items
transactions = {}
for idx, row in df.iterrows():
    url = row['video_url']
    brands = row['mentioned_brands']
    
    # Inicializar si es la primera vez que vemos este URL
    if url not in transactions:
        transactions[url] = set()
    
    # Añadir marcas a la transacción (set garantiza unicidad)
    for brand in brands:
        if brand:  # Ignorar valores vacíos
            transactions[url].add(brand)

# 4. Convertir a formato adecuado para Apriori
transaction_list = list(transactions.values())
unique_brands = set()
for brands in transaction_list:
    unique_brands.update(brands)

# 5. Crear matriz de one-hot encoding (binary representation)
# Cada fila es un video (URL), cada columna es una marca
basket = pd.DataFrame(0, index=range(len(transaction_list)), 
                       columns=sorted(list(unique_brands)))

# Llenar la matriz
for i, brands in enumerate(transaction_list):
    for brand in brands:
        basket.iloc[i][brand] = 1

# 6. Imprimir estadísticas básicas
transactions_with_items = sum(1 for t in transaction_list if len(t) > 0)
multi_brand_transactions = sum(1 for t in transaction_list if len(t) > 1)

print(f"Total de videos (URLs) analizados: {len(transaction_list)}")
print(f"Videos con al menos una marca: {transactions_with_items}")
print(f"Videos con múltiples marcas: {multi_brand_transactions}")
print(f"Total de marcas únicas: {len(unique_brands)}")

# 7. Aplicar algoritmo Apriori
min_support = 0.01  # 1% de los videos
print(f"\nAplicando Apriori con soporte mínimo: {min_support}")

frequent_itemsets = apriori(basket, min_support=min_support, use_colnames=True)

# 8. Generar reglas de asociación
if not frequent_itemsets.empty:
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
    
    if not rules.empty:
        # Ordenar por lift descendente
        rules = rules.sort_values('lift', ascending=False)
        
        # Mostrar top 10 reglas
        print(f"\nSe encontraron {len(rules)} reglas de asociación")
        print("\nTop 10 reglas por lift:")
        
        # Crear dataframe para mejor visualización
        top_rules = pd.DataFrame({
            'Antecedentes': [', '.join(list(x)) for x in rules['antecedents'].head(10)],
            'Consecuentes': [', '.join(list(x)) for x in rules['consequents'].head(10)],
            'Soporte': rules['support'].head(10).round(3),
            'Confianza': rules['confidence'].head(10).round(3),
            'Lift': rules['lift'].head(10).round(3)
        })
        
        print(top_rules)
        
        # Guardar todas las reglas en Excel
        pd.DataFrame({
            'Antecedentes': [', '.join(list(x)) for x in rules['antecedents']],
            'Consecuentes': [', '.join(list(x)) for x in rules['consequents']],
            'Soporte': rules['support'].round(3),
            'Confianza': rules['confidence'].round(3),
            'Lift': rules['lift'].round(3)
        }).to_excel('asociaciones_por_url.xlsx', index=False)
        
        print("\nTodas las reglas guardadas en 'asociaciones_por_url.xlsx'")
    else:
        print("No se encontraron reglas de asociación significativas.")
else:
    print("No se encontraron conjuntos de items frecuentes.")
    
    # Intentar con soporte más bajo
    min_support = 0.005  # 0.5%
    print(f"\nProbando con soporte más bajo: {min_support}")
    
    frequent_itemsets = apriori(basket, min_support=min_support, use_colnames=True)
    
    if not frequent_itemsets.empty:
        rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
        
        if not rules.empty:
            rules = rules.sort_values('lift', ascending=False)
            
            pd.DataFrame({
                'Antecedentes': [', '.join(list(x)) for x in rules['antecedents']],
                'Consecuentes': [', '.join(list(x)) for x in rules['consequents']],
                'Soporte': rules['support'].round(3),
                'Confianza': rules['confidence'].round(3),
                'Lift': rules['lift'].round(3)
            }).to_excel('asociaciones_por_url_soporte_bajo.xlsx', index=False)
            
            print(f"\nSe encontraron {len(rules)} reglas con soporte más bajo")
            print("Guardadas en 'asociaciones_por_url_soporte_bajo.xlsx'")
        else:
            print("No se encontraron reglas de asociación incluso con soporte más bajo.")
    else:
        print("No se encontraron conjuntos frecuentes incluso con soporte más bajo.")

# 9. Análisis complementario: Marcas más frecuentes
brand_counts = defaultdict(int)
for brands in transaction_list:
    for brand in brands:
        brand_counts[brand] += 1

# Ordenar y mostrar top 10 marcas
top_brands = sorted(brand_counts.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nMarcas más mencionadas en los videos:")
for brand, count in top_brands:
    print(f"{brand}: {count} videos ({count/len(transaction_list)*100:.1f}%)")

print("\n=== INTERPRETACIÓN ===")
print("Support: % de videos donde las marcas aparecen juntas")
print("Confidence: % de veces que si aparece la primera marca, también aparece la segunda")
print("Lift > 1: Las marcas tienden a aparecer juntas más de lo esperado por casualidad")

Datos cargados: 1146 videos
Total de videos (URLs) analizados: 183
Videos con al menos una marca: 183
Videos con múltiples marcas: 173
Total de marcas únicas: 83

Aplicando Apriori con soporte mínimo: 0.01


C:\Users\sandr\AppData\Local\Temp\ipykernel_28408\1413138042.py:61: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  basket.iloc[i][brand] = 1
c:\Users\sandr\anaconda3\envs\MachineLearning\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py


Se encontraron 446762 reglas de asociación

Top 10 reglas por lift:
                                             Antecedentes  \
398978  Charlotte Tilbury, DUO, Dior Beauty, Huda Beau...   
446664  Rare Beauty by Selena Gomez, MAKEUP BY MARIO, ...   
328165      Drunk Elephant, DUO, Huda Beauty, Dior Beauty   
52012                           Yves Saint Laurent, tarte   
52013                           Yves Saint Laurent, Tarte   
52016                                   tarte, YSL Beauty   
52017                                   YSL Beauty, Tarte   
398957  Drunk Elephant, LANEIGE, Rare Beauty by Selena...   
398975  Charlotte Tilbury, DUO, Rare Beauty by Selena ...   
398976  Charlotte Tilbury, DUO, Rare Beauty by Selena ...   

                                             Consecuentes  Soporte  Confianza  \
398978  Drunk Elephant, Rare Beauty by Selena Gomez, L...    0.011        1.0   
446664  Drunk Elephant, tarte, Charlotte Tilbury, Rare...    0.011        1.0   
328165         R

In [29]:
# Market Basket Analysis con normalización de nombres de marcas
# para evitar problemas con mayúsculas/minúsculas y variaciones

import pandas as pd
import numpy as np
import ast
from collections import defaultdict
import itertools
import re

# 1. Cargar datos
df = pd.read_excel('C:\\Users\\sandr\\Documents\\scrp_tiktok_tfg\\analysis\\nlp\\products and brands detection\\resultados_finales.xlsx')
print(f"Datos cargados: {df.shape[0]} videos")

# 2. Diccionario de normalización de marcas
# Mapea variaciones del nombre a un nombre estándar
brand_mapping = {
    # Normalizaciones basadas en los resultados que observamos
    'MILK MAKEUP': 'Milk Makeup',
    'milk makeup': 'Milk Makeup',
    
    'ILIA': 'Ilia',
    'ilia': 'Ilia',
    
    'GUCCI': 'Gucci',
    'gucci': 'Gucci',
    'Gucci Beauty': 'Gucci',
    'GUCCI BEAUTY': 'Gucci',
    
    'TOM FORD': 'Tom Ford',
    'Tom Ford': 'Tom Ford',
    'Tom Ford Beauty': 'Tom Ford',
    'TOM FORD BEAUTY': 'Tom Ford',
    
    'Armani Beauty': 'Giorgio Armani Beauty',
    'ARMANI BEAUTY': 'Giorgio Armani Beauty',
    'Giorgio Armani': 'Giorgio Armani Beauty',
    
    'HUDA BEAUTY': 'Huda Beauty',
    'huda beauty': 'Huda Beauty',
    'Huda': 'Huda Beauty',
    'HUDA': 'Huda Beauty',
    
    'DRUNK ELEPHANT': 'Drunk Elephant',
    'drunk elephant': 'Drunk Elephant',
    
    'LANEIGE': 'Laneige',
    'laneige': 'Laneige',
    
    'CHARLOTTE TILBURY': 'Charlotte Tilbury',
    'charlotte tilbury': 'Charlotte Tilbury',
    
    'RARE BEAUTY': 'Rare Beauty',
    'Rare Beauty by Selena Gomez': 'Rare Beauty',
    
    'FENTY BEAUTY': 'Fenty Beauty',
    'Fenty': 'Fenty Beauty',
    'Fenty Beauty by Rihanna': 'Fenty Beauty',
    
    'MAC': 'MAC Cosmetics',
    'Mac': 'MAC Cosmetics',
    'MAC Cosmetics': 'MAC Cosmetics',
    
    'YSL': 'Yves Saint Laurent',
    'YSL Beauty': 'Yves Saint Laurent',
    'Yves Saint Laurent': 'Yves Saint Laurent',
    
    'tarte': 'Tarte',
    'TARTE': 'Tarte',
    'Tarte': 'Tarte',
    
    'NARS': 'NARS',
    'nars': 'NARS',
    
    'DUO': 'DUO',
    'duo': 'DUO',
}

# 3. Función para normalizar nombres de marcas
def normalize_brand(brand):
    if not brand or not isinstance(brand, str):
        return None
        
    # Comprobar si la marca está en nuestro diccionario
    if brand in brand_mapping:
        return brand_mapping[brand]
    
    # Si no está en el diccionario, normalizar basado en reglas generales
    # Eliminar espacios extra y convertir a title case (primera letra mayúscula)
    normalized = re.sub(r'\s+', ' ', brand).strip().title()
    
    # Excepciones para marcas que deben estar en mayúsculas
    if normalized.upper() in ['MAC', 'NARS', 'IT', 'YSL', 'SPF']:
        return normalized.upper()
        
    return normalized

# 4. Convertir strings a listas
def parse_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    elif isinstance(x, list):
        return x
    else:
        return []

df['mentioned_brands'] = df['mentioned_brands'].apply(parse_list)

# 5. Agrupar marcas por URL de video (con normalización)
video_brands = defaultdict(set)
for idx, row in df.iterrows():
    url = row['video_url']
    brands = row['mentioned_brands']
    
    for brand in brands:
        normalized_brand = normalize_brand(brand)
        if normalized_brand:  # Añadir solo si es válido
            video_brands[url].add(normalized_brand)

# 6. Filtrar videos con al menos 2 marcas
multi_brand_videos = {url: brands for url, brands in video_brands.items() if len(brands) >= 2}
print(f"Videos con múltiples marcas (después de normalización): {len(multi_brand_videos)} de {len(video_brands)} videos")

# 7. Calcular frecuencia de cada marca individual
brand_counts = defaultdict(int)
for brands in multi_brand_videos.values():
    for brand in brands:
        brand_counts[brand] += 1

# 8. Calcular frecuencia de pares de marcas
pair_counts = defaultdict(int)
for brands in multi_brand_videos.values():
    # Generar todos los pares posibles de marcas en este video
    for pair in itertools.combinations(sorted(brands), 2):
        # Asegurar que el par esté ordenado alfabéticamente para consistencia
        sorted_pair = tuple(sorted(pair))
        pair_counts[sorted_pair] += 1

# 9. Calcular métricas para cada par
total_videos = len(multi_brand_videos)
pair_metrics = []

for pair, count in pair_counts.items():
    brand1, brand2 = pair
    
    # Soporte: % de videos donde ambas marcas aparecen
    support = count / total_videos
    
    # Confianza en ambas direcciones
    confidence_1_2 = count / brand_counts[brand1]  # P(B|A)
    confidence_2_1 = count / brand_counts[brand2]  # P(A|B)
    
    # Lift: P(A,B) / (P(A) * P(B))
    lift = (count / total_videos) / ((brand_counts[brand1] / total_videos) * (brand_counts[brand2] / total_videos))
    
    pair_metrics.append({
        'Marca1': brand1,
        'Marca2': brand2,
        'Frecuencia': count,
        'Soporte': support,
        'Confianza_1_2': confidence_1_2,
        'Confianza_2_1': confidence_2_1,
        'Lift': lift
    })

# 10. Convertir a DataFrame y ordenar por lift
results_df = pd.DataFrame(pair_metrics)
results_df = results_df.sort_values('Lift', ascending=False)

# 11. Guardar resultados
results_df.to_excel('pares_marcas_normalizado.xlsx', index=False)
print(f"Resultados guardados en 'pares_marcas_normalizado.xlsx'")

# Guardar también las frecuencias individuales
brand_freq_df = pd.DataFrame({
    'Marca': list(brand_counts.keys()),
    'Frecuencia': list(brand_counts.values()),
    'Porcentaje': [count/total_videos*100 for count in brand_counts.values()]
}).sort_values('Frecuencia', ascending=False)

brand_freq_df.to_excel('frecuencia_marcas.xlsx', index=False)
print(f"Frecuencias de marcas guardadas en 'frecuencia_marcas.xlsx'")

# 12. Mostrar top 20 pares con mayor lift
print("\nTop 20 pares de marcas que aparecen juntas (por lift):")
print("-" * 80)
for idx, row in results_df.head(20).iterrows():
    print(f"{row['Marca1']} + {row['Marca2']}:")
    print(f"  Frecuencia: {row['Frecuencia']} videos")
    print(f"  Soporte: {row['Soporte']:.3f} ({row['Soporte']*100:.1f}% de los videos)")
    print(f"  Confianza {row['Marca1']} → {row['Marca2']}: {row['Confianza_1_2']:.2f}")
    print(f"  Confianza {row['Marca2']} → {row['Marca1']}: {row['Confianza_2_1']:.2f}")
    print(f"  Lift: {row['Lift']:.2f}")
    print("-" * 40)

# 13. Mostrar top 10 marcas individuales
print("\nMarcas más mencionadas (después de normalización):")
for brand, count in sorted(brand_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{brand}: {count} videos ({count/total_videos*100:.1f}%)")

print("\n=== INTERPRETACIÓN ===")
print("- Soporte: % de videos donde ambas marcas aparecen juntas")
print("- Confianza_1_2: % de veces que la Marca2 aparece cuando se menciona la Marca1")
print("- Confianza_2_1: % de veces que la Marca1 aparece cuando se menciona la Marca2")
print("- Lift > 1: Las marcas tienden a aparecer juntas más de lo esperado por casualidad")

Datos cargados: 1146 videos
Videos con múltiples marcas (después de normalización): 171 de 183 videos
Resultados guardados en 'pares_marcas_normalizado.xlsx'
Frecuencias de marcas guardadas en 'frecuencia_marcas.xlsx'

Top 20 pares de marcas que aparecen juntas (por lift):
--------------------------------------------------------------------------------
Gucci + Tom Ford:
  Frecuencia: 1 videos
  Soporte: 0.006 (0.6% de los videos)
  Confianza Gucci → Tom Ford: 1.00
  Confianza Tom Ford → Gucci: 0.50
  Lift: 85.50
----------------------------------------
Iconic London + The Ordinary:
  Frecuencia: 1 videos
  Soporte: 0.006 (0.6% de los videos)
  Confianza Iconic London → The Ordinary: 0.33
  Confianza The Ordinary → Iconic London: 1.00
  Lift: 57.00
----------------------------------------
Caudalie + Dermalogica:
  Frecuencia: 2 videos
  Soporte: 0.012 (1.2% de los videos)
  Confianza Caudalie → Dermalogica: 1.00
  Confianza Dermalogica → Caudalie: 0.67
  Lift: 57.00
--------------------

In [22]:
#Interpretación: 
# Antecedents → Productos/Marcas que aparecen juntos.
# Consequents → Productos/Marcas que aparecen juntos.
# Support → Frecuencia con la que aparecen juntos.
# Confidence → Probabilidad de que si aparece el antecedente, aparezca el consecuente.
# Lift → Probabilidad de que aparezcan juntos, en comparación con si fueran independientes.
    # Si el lift es mayor a 1, significa que hay una correlación positiva entre los productos/marcas.
    # Si el lift es menor a 1, significa que hay una correlación negativa entre los productos/marcas.
    # Si el lift es igual a 1, significa que no hay correlación entre los productos/marcas. 